In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
torch.set_default_dtype(torch.float32)

from src.cann.models.minimal import Minimal
from src.cann.models.polykan import PolyKAN
from src.cann.models.base import ActNN
from src.cann.train import ACT_DICT

In [ ]:
# make input grid, covering B/S neighborhoods from 0 through 8
n_inputs = torch.ones(200,1,3,3)

halfway = 100
dn = 9 / (halfway-1)
n_inputs[:halfway] *= torch.arange(0, 9+dn, dn).reshape(-1,1,1,1)

#for coef, xx in enumerate(range(0,90,10)):
#    n_inputs[xx:xx+10] *= (coef) 

n_inputs[halfway:] = n_inputs[:halfway]
n_inputs /= 8 
n_inputs[:halfway,:,1,1] *= 0.0

n_inputs[halfway:,:,1,1] = 1.0

In [ ]:
entry["rulestring"].split("/")[0][1:]

In [ ]:

my_cmap = plt.get_cmap("plasma")

color_index = 32

results_dir = os.path.join("..","results")
list_dir = os.listdir(results_dir)

for elem in zip(list_dir):
    if (elem[0].endswith("l1_1") or elem[0].endswith("l1_m")) and "knockout" in elem[0]:

        df = pd.read_csv(os.path.join(results_dir, elem[0], "exp.csv"))
        if (df["final_grid_accuracy"] == 1.0).sum():
            
            entry = df.loc[df[df["final_grid_accuracy"] == 1.0].index[0]] 
            width = entry["model_width"]
            depth = entry["model_depth"]
                            
            run_df = pd.read_csv(os.path.join("..", entry["run_filename"]))
            params = np.load(os.path.join("..", run_df["parameters_filename"][len(df2)-1]))[-1]
                             
            if entry["model_name"] == "PolyKAN":
                print("polykan")
                model = PolyKAN(width=width, depth=depth)
            else:
                activation = ACT_DICT[entry["activation_name"].lower()]
                model = ActNN(width=width, depth=depth, activation=activation,\
                             )

            model.set_parameters(params)
        
            with torch.no_grad():
                
                bs_out = model(n_inputs)[:,:,1,1]
            
            fig, ax = plt.subplots(1,1) #figure()
            ax.plot(bs_out[:halfway], "-", alpha=0.5, lw=3, color=my_cmap(color_index), label="B")
            ax.plot(bs_out[halfway:], "-", alpha=0.5, lw=3, color=my_cmap(255-color_index),label="S")
            
            ax.plot(bs_out[:halfway].round(), "--", alpha=0.25, lw=3, color=my_cmap(color_index), label="B(round)")
            ax.plot(bs_out[halfway:].round(), "--", alpha=0.25, lw=3, color=my_cmap(255-color_index),label="S(rounded)")
            
            moores = n_inputs[:halfway].sum((1,2,3))
            x_ticks = np.arange(0,halfway,10)
            x_ticklabels = moores[::10]

            
            for num, ss in enumerate(entry["rulestring"].split("/")[1][1:]):
                if num:
                    ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(ss)))[0][0], 1,  s=300,\
                               marker="s", alpha=0.1, edgecolors="k", lw=2, color=my_cmap(color_index-50))
                else:
                    ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(ss)))[0][0], 1,  s=300, \
                               marker="s", alpha=0.1, edgecolors="k", lw=2, color=my_cmap(color_index-50), label="S (ground truth)")
            for num, bb in enumerate(entry["rulestring"].split("/")[0][1:]):
                print(int(bb))
                if num:
                    ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(bb)))[0][0], 1, s=300,\
                                alpha=0.3, color=my_cmap(color_index+50))
                else:
                    ax.scatter(torch.where(n_inputs.sum((1,2,3)) == (float(bb)))[0][0], 1, s=300,\
                                alpha=0.3, color=my_cmap(color_index+50), label="B (ground truth)")

            ax.set_xticks([elem+5 for elem in x_ticks])
            ax.set_xticklabels([f"{elem.item():.2f}" for elem in x_ticklabels], rotation=45)
            
            plt.title(f"B/S equivalent for {entry['model_name']} {elem[0]}")
            plt.legend()
            plt.show()

        
            color_index += 32
            

In [ ]:
elem

In [ ]:
model.get_parameters().shape, params.shape, entry["activation_name"]

In [ ]:
entry["mo

In [ ]:
dir(df[df["final_grid_accuracy"] == 1.0])
entry = df[df["final_grid_accuracy"] == 1.0].loc[0]

entry
#df[df["final_grid_accuracy"] == 1.0].index

df2 = pd.read_csv(os.path.join("..", entry["run_filename"]))
df2["parameters_filename"][len(df2)-1]

#.to_list()[-1] #[len(df2)]
                  

In [ ]:

# Minimal ReLU NN from Springer and Kenyon
minimal = Minimal()

# PolyKAN trained with learnable activation function and with learnable neural weights
poly_c1n1_fn = os.path.join("..", "results", "poly_no_knockout_l1_1", "parameters_0.npy")
poly_c1n1_params = np.load(poly_c1n1_fn)
poly_c1n1_model = PolyKAN()
poly_c1n1_model.set_parameters(poly_c1n1_params[-1])

# PolyKAN trained with learnable activation function and without learnable neural weights
poly_c1n0_fn = os.path.join("..", "results", "poly_weight_knockout_neur_l1_1", "parameters_0.npy")
poly_c1n0_params = np.load(poly_c1n0_fn)
poly_c1n0_model = PolyKAN()
poly_c1n0_model.set_parameters(poly_c1n0_params[-1])

# PolyKAN trained without learnable activation function and with learnable neural weights
poly_c0n1_fn = os.path.join("..", "results", "poly_act_knockout_coef_l1_1", "parameters_0.npy")
poly_c0n1_params = np.load(poly_c0n1_fn)
poly_c0n1_model = PolyKAN()
poly_c0n1_model.set_parameters(poly_c0n1_params[-1])

# PReLU trained with learnable activation function and with learnable neural weights
prelu_c1n1_fn = os.path.join("..", "results","prelu_no_knockout_l1_1","parameters_0.npy")
prelu_c1n1_params = np.load(prelu_c1n1_fn)
prelu_c1n1_model = ActNN(activation=ACT_DICT["prelu"])
prelu_c1n1_model.set_parameters(prelu_c1n1_params[-1])

# PReLU trained with learnable activation function and without learnable neural weights
prelu_c1n0_fn = os.path.join("..", "results","prelu_weight_knockout_neur_l1_1","parameters_0.npy")
prelu_c1n0_params = np.load(prelu_c1n0_fn)
prelu_c1n0_model = ActNN(activation=ACT_DICT["prelu"])
prelu_c1n0_model.set_parameters(prelu_c1n0_params[-1])

# PReLU trained without learnable activation function and with learnable neural weights
prelu_c0n1_fn = os.path.join("..", "results","prelu_act_knockout_coef_l1_1","parameters_0.npy")
prelu_c0n1_params = np.load(prelu_c0n1_fn)
prelu_c0n1_model = ActNN(activation=ACT_DICT["prelu"])
prelu_c0n1_model.set_parameters(prelu_c0n1_params[-1])


# adaptive sigmoid trained with learnable activation function and with learnable neural weights
asigmoid_c1n1_fn = os.path.join("..", "results","asigmoid_no_knockout_l1_1","parameters_0.npy")
asigmoid_c1n1_params = np.load(asigmoid_c1n1_fn)
asigmoid_c1n1_model = ActNN(activation=ACT_DICT["asigmoid"])
asigmoid_c1n1_model.set_parameters(asigmoid_c1n1_params[-1])

# adaptive sigmoid trained with learnable activation function and without learnable neural weights
asigmoid_c1n0_fn = os.path.join("..", "results","asigmoid_weight_knockout_neur_l1_1","parameters_0.npy")
asigmoid_c1n0_params = np.load(asigmoid_c1n0_fn)
asigmoid_c1n0_model = ActNN(activation=ACT_DICT["asigmoid"])
asigmoid_c1n0_model.set_parameters(asigmoid_c1n0_params[-1])

# adaptive sigmoid trained without learnable activation function and with learnable neural weights
asigmoid_c0n1_fn = os.path.join("..", "results","asigmoid_act_knockout_coef_l1_1","parameters_0.npy")
asigmoid_c0n1_params = np.load(asigmoid_c0n1_fn)
asigmoid_c0n1_model = ActNN(activation=ACT_DICT["asigmoid"])
asigmoid_c0n1_model.set_parameters(asigmoid_c0n1_params[-1])

models = [minimal,\
          poly_c1n1_model, poly_c1n0_model, poly_c0n1_model,\
          prelu_c1n1_model, prelu_c1n0_model, prelu_c0n1_model,\
         asigmoid_c1n1_model, asigmoid_c1n0_model, asigmoid_c0n1_model]
model_names = ["K&S MinimalGoLNN",\
               "PolyKAN Full",\
               "PolyKAN StaticSynapse",\
               "PolyKAN StaticActivation",\
               "PReLU Full",\
               "PReLU StaticSynapse",\
               "PReLU StaticActivation",\
               "aSigmoid Full",\
               "aSigmoid StaticSynapse",\
               "aSigmoid StaticActivation"
               ]

In [ ]:
df = pd.read_csv(os.path.join("..", "results","asigmoid_act_knockout_coef_l1_1","exp.csv"))

df["final_grid_accuracy"] == 1.0

In [ ]:

my_cmap = plt.get_cmap("plasma")

color_index = 32

for model_name, model in zip(model_names, models):
    with torch.no_grad():
        
        bs_out = model(n_inputs)[:,:,1,1]
    
    fig, ax = plt.subplots(1,1) #figure()
    ax.plot(bs_out[:halfway], "-", alpha=0.5, lw=3, color=my_cmap(color_index), label="B")
    ax.plot(bs_out[halfway:], "-", alpha=0.5, lw=3, color=my_cmap(255-color_index),label="S")
    
    ax.plot(bs_out[:halfway].round(), "--", alpha=0.25, lw=3, color=my_cmap(color_index), label="B(round)")
    ax.plot(bs_out[halfway:].round(), "--", alpha=0.25, lw=3, color=my_cmap(255-color_index),label="S(rounded)")
    
    moores = n_inputs[:halfway].sum((1,2,3))
    x_ticks = np.arange(0,halfway,10)
    x_ticklabels = moores[::10]

    print(x_ticks, x_ticklabels)
    ax.set_xticks([elem+5 for elem in x_ticks])
    ax.set_xticklabels([f"{elem.item():.2f}" for elem in x_ticklabels], rotation=45)
    
    plt.title(f"B/S equivalent for {model_name}")
    plt.legend()
    plt.show()

    color_index += 32


In [ ]:
# make input grid, covering B/S neighborhoods from 0 through 8
n_inputs = torch.ones(180,1,3,3)
halfway = 90

for coef, xx in enumerate(range(0,90,10)):
    n_inputs[xx:xx+10] *= (coef) 

n_inputs[halfway:] = n_inputs[:halfway]
n_inputs /= 8 
n_inputs[:halfway,:,1,1] *= 0.0

n_inputs[halfway:,:,1,1] = 1.0

In [ ]:


minimal = Minimal()
models = [minimal]
model_names = ["Game of Life"]

my_cmap = plt.get_cmap("plasma")

color_index = 32

for model_name, model in zip(model_names, models):
    with torch.no_grad():
        model = Minimal()
        
        bs_out = model(n_inputs)[:,:,1,1]
    
    fig, ax = plt.subplots(1,1) #figure()
    ax.plot(bs_out[:halfway], "--", alpha=0.5, lw=3, color=my_cmap(color_index), label="B")
    ax.plot(bs_out[halfway:], "-", alpha=0.5, lw=3, color=my_cmap(255-color_index),label="S")
    moores = n_inputs[:halfway].sum((1,2,3))
    x_ticks = np.arange(0,halfway,10)
    x_ticklabels = moores[::10]

    ax.set_xticks([elem+5 for elem in x_ticks])
    ax.set_xticklabels([int(elem.item()) for elem in x_ticklabels], rotation=45)
    
    plt.title(f"B/S equivalent for {model_name}")
    plt.legend()
    plt.show()

    color_index += 32


In [ ]:
moores.shape, moores[::10].shape, np.arange(0,90,10).shape